# Practice workflow (mock of the hackathon template)
Run cells top to bottom. Set `LLM_PROVIDER=mock` in the first cell to test without spending tokens.

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.path.abspath('main.py')))
sys.path.insert(0, '.')
# os.environ['LLM_PROVIDER'] = 'mock'   # uncomment: no LLM calls
import pandas as pd
pd.set_option('display.width', 200)

## 1. Explore the data (point-in-time)

In [ ]:
from src.utils import data, pit
meta = data.read_assets_metadata(); meta

In [ ]:
as_of = '2025-10-31'
lv = pit.assets_asof(as_of)
(lv.iloc[-252:] / lv.iloc[-252]).plot(figsize=(12, 5), legend=False, title=f'12m to {as_of}');

In [ ]:
from src.agents.tools import market_indicators_table, macro_snapshot_table, recent_news_text
print(market_indicators_table(as_of))
print(macro_snapshot_table(as_of))
print(recent_news_text(as_of, 7))

## 2. Quant core: risk of the strategic benchmark

In [ ]:
from src.config import STRATEGIC_BENCHMARK, LOOKBACK_DAYS
from src.utils.analytics import portfolio_metrics, check_constraints
b = pd.Series(STRATEGIC_BENCHMARK)
rets = pit.returns_base_ccy_asof(as_of, LOOKBACK_DAYS)
print(check_constraints(b.reindex(meta.ticker).fillna(0), meta))
portfolio_metrics(b, rets, meta)

## 3. Run the agents

In [ ]:
from src.agents.graph import build_workflow
from src.core.session import start_session
from src.core.run import run_date
from src.core.results import write_weights
workflow = build_workflow()
workflow.get_graph().draw_mermaid()  # paste into mermaid.live to see the graph

In [ ]:
session = start_session()
result = run_date(workflow, as_of, session)
print(result['analyst_summary'][:2000])

In [ ]:
pd.DataFrame({'weight': result['weights'], 'benchmark': STRATEGIC_BENCHMARK}).sort_values('weight', ascending=False)

In [ ]:
print(result['risk_commentary'])
result['adjustments']

In [ ]:
# after running all REPORTING_DATES:
# from src.config import REPORTING_DATES
# for d in REPORTING_DATES: run_date(workflow, d, session)
# write_weights(session)